# 02 · Feature engineering for a fixed model (drill)

**Style B: blind-code drill.** Write each piece from memory into the empty cell, run the check, then delete
your code. The checks only use the function names in each spec.
Reference: `../solutions/drills/02_feature_engineering.py`.

**Traces to** `notes/ioai-task-patterns.md` P5 · T24-O-HYPER (symmetry augmentation + PCA for a frozen
`LinearRegression`), T24-H-ML (signal arrays, depth-limited tree, "the data itself was corrupted").

Total suggested time: **50 min**.

In [ ]:
import itertools
import numpy as np
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.datasets import load_digits, make_friedman1
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
rng = np.random.default_rng(123)

## 1. Image summary features ⏱ 8 min

`image_features(imgs) -> F`: for `imgs` of shape `(n, H, W)` return a 2-D float array `(n, k)` with
`k ≤ 40` and no NaNs. Include at least row sums, column sums and a left-right asymmetry score.
The check trains a **fixed** `DecisionTreeClassifier(max_depth=4)` on digits. Raw pixels score 0.58, and you
need ≥ 0.63. More columns are not automatically better for a greedy depth-4 tree.

In [ ]:
d = load_digits()
a, b, ya, yb = train_test_split(d.images, d.target, test_size=0.3, stratify=d.target, random_state=0)
Fa, Fb = image_features(a), image_features(b)
assert Fa.ndim == 2 and Fa.shape[0] == len(a) and Fa.shape[1] <= 40 and not np.isnan(Fa).any()
acc = DecisionTreeClassifier(max_depth=4, random_state=0).fit(Fa, ya).score(Fb, yb)
assert acc >= 0.63, acc
print(f"section 1 ok (fixed tree accuracy {acc:.3f})")

## 2. Channel-permutation-invariant features ⏱ 8 min

Sensor channels arrive in a random order. `invariant_features(X) -> F` maps `(n, C, T)` signals to `(n, k)`
features that are **identical** whatever the channel order. Hint: statistics per channel, then a symmetric
reduction over channels (mean / max / sorted values).

In [ ]:
X = rng.normal(size=(50, 6, 64)) * rng.uniform(0.5, 2, size=(50, 6, 1))
F = invariant_features(X)
perm = rng.permutation(6)
assert F.ndim == 2 and F.shape[0] == 50
assert np.allclose(F, invariant_features(X[:, perm])), "features changed when channels were shuffled"
assert not np.allclose(F[0], F[1]), "features should still differ between samples"
print("section 2 ok")

## 3. Dominant frequency ⏱ 8 min

`dominant_freq(X, fs) -> f`: for `X` of shape `(n, T)` sampled at `fs` Hz, return `(n,)` estimates of each
signal's dominant frequency in Hz. Ignore the DC bin. Must be within 0.2 Hz on average.

In [ ]:
fs, T = 50, 200
f_true = rng.uniform(1, 20, 400)
t = np.arange(T) / fs
Xs = 2 * np.sin(2 * np.pi * f_true[:, None] * t + rng.uniform(0, 6, (400, 1))) + 0.5 + 0.3 * rng.normal(size=(400, T))
f_hat = np.asarray(dominant_freq(Xs, fs))
assert f_hat.shape == (400,)
err = np.abs(f_hat - f_true).mean()
assert err < 0.2, err
print(f"section 3 ok (mean error {err:.3f} Hz)")

## 4. PCA features without leakage ⏱ 6 min

`pca_features(Xtr, Xte, k) -> (Ztr, Zte)`: fit PCA with `k` components on `Xtr` **only** and transform
both sets.

In [ ]:
Xtr, Xte = rng.normal(size=(200, 30)) @ rng.normal(size=(30, 30)), rng.normal(size=(80, 30))
Ztr, Zte = pca_features(Xtr, Xte, 5)
ref = PCA(n_components=5).fit(Xtr)
# component signs may flip between implementations, so compare absolute projections
assert Ztr.shape == (200, 5) and Zte.shape == (80, 5)
assert np.allclose(np.abs(Zte), np.abs(ref.transform(Xte)), atol=1e-4), "test set must use the train-fitted PCA"
print("section 4 ok")

## 5. All-axis-permutation augmentation ⏱ 8 min

`augment_axes(X, y) -> (X_aug, y_aug)`: `X` has shape `(n, A, A, A)`. Return all 6 permutations of the three
spatial axes for every sample (`6n` samples), with correctly aligned labels.

In [ ]:
X = rng.normal(size=(7, 3, 3, 3)); y = np.arange(7.0)
Xa, ya = augment_axes(X, y)
assert Xa.shape == (42, 3, 3, 3) and ya.shape == (42,)
for i in range(7):
    copies = [Xa[j] for j in range(42) if ya[j] == y[i]]
    assert len(copies) == 6
    want = {X[i].transpose(p).tobytes() for p in itertools.permutations(range(3))}
    assert {c.tobytes() for c in copies} == want, "labels do not line up with their permuted copies"
print("section 5 ok")

## 6. Beat a frozen `LinearRegression` on Friedman #1 ⏱ 12 min

`friedman_features(X) -> F` for `make_friedman1` inputs `(n, 10)`, with at most 12 columns. A fixed
`LinearRegression()` on your features needs R² ≥ 0.95 on held-out data.

In [ ]:
X, y = make_friedman1(n_samples=3000, n_features=10, noise=0.5, random_state=1)
a, b, ya, yb = train_test_split(X, y, test_size=0.3, random_state=1)
Fa, Fb = friedman_features(a), friedman_features(b)
assert Fa.shape[1] <= 12
r2 = r2_score(yb, LinearRegression().fit(Fa, ya).predict(Fb))
assert r2 >= 0.95, r2
print(f"section 6 ok (R² {r2:.4f})")